In [3]:
import random
import json
import pickle
import tensorflow as tf
import keras
import nltk
nltk.download('all')
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping

import numpy as np

lemmatizer = WordNetLemmatizer()

intents = json.loads(open("/content/sample_data/intents_medquad.json").read())

words = []
classes = []
documents = []

ignore_letters = ["?", "!", ".", ","]

for intent in intents["intents"]:
    for pattern in intent["patterns"]:
        word_list = nltk.word_tokenize(pattern)
        words.extend(word_list)
        documents.append((word_list, intent["tag"]))

        if intent["tag"] not in classes:
            classes.append(intent["tag"])
words = [lemmatizer.lemmatize(word)
         for word in words if word not in ignore_letters]

words = sorted(set(words))
classes = sorted(set(classes))

pickle.dump(words, open('words.pkl', 'wb'))
pickle.dump(classes, open('classes.pkl', 'wb'))

dataset = []
template = [0]*len(classes)

for document in documents:
    bag = []
    word_patterns = document[0]
    word_patterns = [lemmatizer.lemmatize(word.lower())
                     for word in word_patterns]

    for word in words:
        bag.append(1) if word in word_patterns else bag.append(0)

    output_row = list(template)
    output_row[classes.index(document[1])] = 1
    dataset.append([bag, output_row])

random.shuffle(dataset)
dataset = np.array(dataset, dtype=object)

train_x = list(dataset[:, 0])
train_y = list(dataset[:, 1])

model = Sequential()
model.add(Dense(256, input_shape=(len(train_x[0]),),
                activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))


sgd = SGD(learning_rate=0.01,
          momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy',
              optimizer=sgd, metrics=['accuracy'])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

hist = model.fit(np.array(train_x),np.array(train_y), epochs=100, batch_size=8,
    validation_split=0.2, callbacks=[early_stop],verbose=1)

model.save("chatbot_model.keras", hist)
print("Done!")

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_r

Epoch 1/100
1497/1497 ━━━━━━━━━━━━━━━━━━━━ 32s 20ms/step - accuracy: 6.6800e-04 - loss: 8.4507 - val_accuracy: 6.6778e-04 - val_loss: 8.4116
Epoch 2/100
1497/1497 ━━━━━━━━━━━━━━━━━━━━ 30s 20ms/step - accuracy: 9.1850e-04 - loss: 8.3036 - val_accuracy: 3.3389e-04 - val_loss: 8.2645
Epoch 3/100
1497/1497 ━━━━━━━━━━━━━━━━━━━━ 30s 20ms/step - accuracy: 0.0014 - loss: 8.0964 - val_accuracy: 0.0020 - val_loss: 8.0454
Epoch 4/100
1497/1497 ━━━━━━━━━━━━━━━━━━━━ 30s 20ms/step - accuracy: 0.0020 - loss: 7.8144 - val_accuracy: 0.0027 - val_loss: 7.7648
Epoch 5/100
1497/1497 ━━━━━━━━━━━━━━━━━━━━ 30s 20ms/step - accuracy: 0.0049 - loss: 7.5398 - val_accuracy: 0.0053 - val_loss: 7.5001
Epoch 6/100
1497/1497 ━━━━━━━━━━━━━━━━━━━━ 30s 20ms/step - accuracy: 0.0099 - loss: 7.2673 - val_accuracy: 0.0107 - val_loss: 7.2055
Epoch 7/100
1497/1497 ━━━━━━━━━━━━━━━━━━━━ 32s 21ms/step - accuracy: 0.0173 - loss: 6.9359 - val_accuracy: 0.0240 - val_loss: 6.9020
Epoch 8/100
1497/1497 ━━━━━━━━━━━━━━━━━━━━ 30s 20ms/s

In [ ]:
!python chatbot.py


2026-04-17 10:33:16.780909: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776421996.805569   31274 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776421996.812781   31274 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776421996.830312   31274 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776421996.830379   31274 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776421996.830395   31274 computation_placer.cc:177] computation placer alr